In [19]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
from daemon_analysis_tools.io.csv_handler import load_and_process_csv
from daemon_analysis_tools.io.yaml_handler import save_answers_to_yaml, load_answers_from_yaml
from daemon_analysis_tools.processing.grouper import group_questions_by_journal
from daemon_analysis_tools.services.discrepancy_resolver import resolve_discrepancy

Load and process data:
- Group answers by publisher and journal, trying to uniform names written in slightly different ways.
- Store in a DataFrame

In [21]:
data = load_and_process_csv("../../data/raw/rdp.csv")

Get a `dict` labeled by publisher names of `dict`s labeled by journal names of `dict`s of `Question` instances. The `.answer` attribute contains the answers given by the respondents and the explanations text to motivate it.

In [22]:
question_metadata_file = "../../data/metadata/question_metadata.yaml"

grouped_questions = group_questions_by_journal(data, question_metadata_file)

## Resolve discrepancies

The `Question` class has a `.resolve_discrepancies` method which updates `Question.anwsers` with the correct answer.

For example, let's consider IOP's 2D Materials. Question 7 has discrepancies.

In [23]:
for journal, data in grouped_questions["Frontiers"].items():
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies():
            answer.print_qa()
    print("\n\n")

frontiers_in_bioengineering_and_biotechnology
3. Data sharing requirements in RDP
  Resp. 0:
    Answer: Public data sharing of all data required.
    Explanation: Frontiers requires that authors make the "minimal data set" underlying the findings described and used to reach the conclusions of the manuscript, available to any qualified researchers. However, exceptions are granted if data cannot be made publicly available for legal or ethical reasons.
  Resp. 1:
    Answer: Data sharing required but not publicly (e.g. available upon request is allowed).
    Explanation: Frontiers requires that authors make the "minimal data set" underlying the findings described and used to reach the conclusions of the manuscript, available to any qualified researchers.
5. Citability and findability of data 
  Resp. 0:
    Answer: DOIs or other persistent identifiers recommended for datasets or codes.
    Explanation: Authors are encouraged to cite all datasets generated or analyzed in the study. Where 

Inconsistencies can be removed manually, passing the index of the correct respondent.

In [24]:
for j in ["frontiers_in_bioengineering_and_biotechnology",
          "frontiers_in_chemical_engineering",
          "frontiers_in_chemistry",
          "frontiers_in_energy_research",
          "frontiers_in_materials",
          "frontiers_in_nanotechnology",
          "frontiers_in_physics",
          ]:
    for i in [3, 5, 10]:
        resolve_discrepancy(
            grouped_questions["Frontiers"][j][i],
            correct_answer=0,
            discrepancy_reason="Language understanding",
        )
    
    for i in [7, 8, 15, 16]:
        resolve_discrepancy(
            grouped_questions["Frontiers"][j][i],
            correct_answer=0,
            discrepancy_reason="Text not found",
        )

    for i in [13]:
        resolve_discrepancy(
            grouped_questions["Frontiers"][j][i],
            correct_answer=1,
            discrepancy_reason="Text not found",
        )

In [25]:
for journal, data in grouped_questions["Frontiers"].items():
    print("#############################################################")
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies() and answer.correct_answer is None:
            answer.print_qa()

#############################################################
frontiers_in_bioengineering_and_biotechnology
#############################################################
frontiers_in_chemical_engineering
#############################################################
frontiers_in_chemistry
#############################################################
frontiers_in_energy_research
#############################################################
frontiers_in_materials
#############################################################
frontiers_in_nanotechnology
#############################################################
frontiers_in_physics


In [26]:
save_answers_to_yaml(
    grouped_questions,
    parent_folder="../../data/processed/all_answers",
    save_only=["Frontiers"],
)

After doing this, the `.get_final_answer()` method returns the correct answer.